In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pandas
!pip install requests==2.32.3
!pip install langchain-text-splitters
!pip install -q -U bitsandbytes accelerate peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. MIMIC 데이터 로드
file_path = "/content/drive/MyDrive/DILAB/MARS/mimic-iv-note_2.2/files/note/discharge.csv"

# 1000개만 무작위 추출
print("데이터 로드 중")
df = pd.read_csv(file_path, nrows=5000)
df_sample = df.sample(1000, random_state=42) # 1000개 샘플링
print(f"데이터 로드 완료: {len(df_sample)}건의 퇴원 요약지 사용")

print(df["text"][0])

데이터 로드 중
데이터 로드 완료: 1000건의 퇴원 요약지 사용
 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
distension over past week.  
Pt reports self-discontinuing lasix and spirnolactone ___ weeks 
ago, because she feels like "they don't do anything" and that 
she "doesn't want to put more chemicals in her." She does not 
follow Na-restricted diets. In the past week, she notes that she 
has been having worsening abd distension and discomfort. She 
denies ___ edema, or SOB, or orthopnea. She denies f/c/n/v, d/c, 
dysuria. She had food poisoning a week ago from ea

In [9]:
def extract_core_sections(text):
    # 추출할 섹션 헤더 정의 (MIMIC 데이터에서 주로 쓰이는 헤더들)
    headers = {
        "CC": ["Chief Complaint:", "Chief Complaint"],
        "HPI": ["History of Present Illness:", "HISTORY OF PRESENT ILLNESS:"],
        "Dx": ["Discharge Diagnosis:", "DISCHARGE DIAGNOSIS:"],
        "Proc": ["Major Surgical or Invasive Procedure:", "Major Surgical or Invasive Procedure"],
        "BHC": ["Brief Hospital Course:", "HOSPITAL COURSE:", "BRIEF HOSPITAL COURSE:"]
    }

    extracted_text = []

    # 각 섹션별로 내용을 찾아서 추출
    for key, patterns in headers.items():
        section_content = ""
        for pattern in patterns:
            # 정규표현식으로 헤더 뒤의 내용 찾기
            try:
                # 헤더 위치 찾기
                start_idx = text.find(pattern)
                if start_idx != -1:
                    start_content = start_idx + len(pattern)
                    temp_text = text[start_content:]
                    end_idx = temp_text.find("\n \n")
                    if end_idx == -1: end_idx = temp_text.find("\n\n")
                    if end_idx == -1: end_idx = 500

                    section_content = temp_text[:end_idx].strip()
                    break # 하나 찾으면 루프 탈출
            except:
                continue

        if section_content:
            # "헤더: 내용" 형식으로 저장
            extracted_text.append(f"{key}: {section_content}")

    return "\n".join(extracted_text)

In [10]:
from langchain_community.vectorstores import FAISS

print("핵심 섹션 추출 및 벡터화 진행 중...")

documents = []
for idx, row in df_sample.iterrows():
    # 1) 핵심 내용만 추출 (Text Cleaning)
    cleaned_text = extract_core_sections(str(row["text"]))

    # 2) 메타데이터 저장
    meta = {
        "subject_id": row.get("subject_id", "unknown"),
        "original_text_preview": str(row["text"])[:200]
    }

    doc = Document(page_content=cleaned_text, metadata=meta)
    documents.append(doc)

# 청킹 (Chunking)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splitted_docs = text_splitter.split_documents(documents)

# 임베딩 & FAISS 저장
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splitted_docs, embeddings)

print(f"벡터 DB 구축 완료! (총 {len(splitted_docs)}개 청크)")

핵심 섹션 추출 및 벡터화 진행 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

벡터 DB 구축 완료! (총 6330개 청크)


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 테스트 질문
query = input()

print(f"\n검색 질의: {query}")
print("="*60)

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"[검색 결과 {i+1}] (환자 ID: {doc.metadata["subject_id"]})")
    print("-" * 30)
    print(doc.page_content)
    print("="*60)

Sudden onset of right-sided weakness and slurred speech.

검색 질의: Sudden onset of right-sided weakness and slurred speech.
[검색 결과 1] (환자 ID: 10017041)
------------------------------
CC: slurred speech
HPI: History of Present Illness (per Dr. ___:

The pt is a ___ Left handed woman who presents as a
code stroke. She was in normal state of health when at 10 pm she
suddenly developed acute onset of slurred speech. Along with 
this
she states that she felt as though her whole left side of her
body felt week from her arm to foot. These symptoms lasted about
___ min and resolved on there own except she still thinks her
left arm is weak. This was witnessed by a friend who notified
family first.
[검색 결과 2] (환자 ID: 10034742)
------------------------------
CC: gait instability
HPI: HPI: ___ is a ___ F with a history of chronic back pain
s/p multiple lumar surgeries and hypothyroidism who is
transferred from ___ where she presented with 4
days of maliase, unsteady gait and slurring of speech which


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

adapter_path = "/content/drive/MyDrive/DILAB/Qwen2-7B-Instruct/adapter/original_1.4"

# 베이스 모델
base_model_id = "Qwen/Qwen2-7B-Instruct"

print(f"베이스 모델: {base_model_id}")
print(f"어댑터 경로: {adapter_path}")

# 토크나이저 & 베이스 모델 로드 (4비트 양자화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 베이스 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("로컬 어댑터를 베이스 모델에 적용 중...")
try:
    model = PeftModel.from_pretrained(base_model, adapter_path)
except Exception as e:
    print(f"어댑터 로드 실패: {e}")
    model = base_model

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


베이스 모델: Qwen/Qwen2-7B-Instruct
어댑터 경로: /content/drive/MyDrive/DILAB/Qwen2-7B-Instruct/adapter/original_1.4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [ ]:
!pip install -q --upgrade --force-reinstall bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.9/520.9 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━